In [101]:
import yfinance as yf
import pandas as pd
import numpy as np
import subprocess
import sys

In [102]:
print(f"Current version: {yf.__version__}")

Current version: 1.3.0


In [103]:
import pandas as pd
import yfinance as yf


def pull_yf_ticker_data(ticker_list: list):
    """Loops through a list of tickers and pulls all available annual data

    from yahoo finance. Returns a multi-row pandas DataFrame structured
    by Ticker and Year.
    """
    extracted_data = []

    for ticker_str in ticker_list:
        print(f"Fetching historical data for: {ticker_str}...")
        try:
            ticker = yf.Ticker(ticker_str)
            info = ticker.info

            # 1. Fetch metadata & currencies
            trading_curr = info.get("currency", "")
            if trading_curr:
                trading_curr = trading_curr.upper()

            financial_curr = info.get("financialCurrency", "")
            if financial_curr:
                financial_curr = financial_curr.upper()

            market_cap = info.get("marketCap", None)

            # 2. Convert live Market Cap if currencies don't match
            market_cap_converted = market_cap
            if (
                trading_curr
                and financial_curr
                and trading_curr != financial_curr
            ):
                fx_ticker_str = f"{trading_curr}{financial_curr}=X"
                try:
                    fx_data = yf.Ticker(fx_ticker_str).history(period="1d")
                    if not fx_data.empty:
                        fx_rate = fx_data["Close"].iloc[-1]
                        if market_cap:
                            market_cap_converted = market_cap * fx_rate
                except Exception:
                    pass

            # 3. Pull Live TTM Net Income (for the current period)
            try:
                ttm_inc = ticker.ttm_income_stmt
                ttm_net_inc = (
                    ttm_inc.loc["Net Income"].iloc[0]
                    if not ttm_inc.empty and "Net Income" in ttm_inc.index
                    else None
                )
            except Exception:
                ttm_net_inc = None

            if ttm_net_inc is None:
                q_inc = ticker.quarterly_income_stmt
                ttm_net_inc = (
                    q_inc.loc["Net Income"].iloc[:4].sum()
                    if not q_inc.empty and "Net Income" in q_inc.index
                    else None
                )

            # 4. Extract ALL Historical Statements
            # We switch to annual balance sheet & annual income statement
            annual_bs = ticker.balance_sheet
            annual_inc = ticker.income_stmt

            if not annual_bs.empty:
                # Loop through every reporting date available (columns)
                for report_date in annual_bs.columns:
                    # Convert Timestamp to a clean string format (YYYY-MM-DD)
                    date_str = str(report_date.date())
                    # Extract the year integer to make downstream filtering easy
                    year_val = report_date.year

                    # Extract balance sheet columns for this specific date
                    bs_col = annual_bs[report_date]

                    total_assets = bs_col.get("Total Assets", None)
                    current_assets = bs_col.get("Current Assets", None)
                    total_liabilities = bs_col.get(
                        "Total Liabilities Net Minority Interest", None
                    )
                    total_investments = bs_col.get(
                        "Investment Properties", None
                    )
                    total_goodwill_intangibles = bs_col.get(
                        "Goodwill And Other Intangible Assets", None
                    )

                    # Extract matching income statement column for this exact date
                    # Note: yfinance matching columns might differ slightly by a day due to weekend alignments,
                    # so we look for the column closest to our balance sheet date.
                    fy_net_inc = None
                    if not annual_inc.empty:
                        # Try exact match first
                        if report_date in annual_inc.columns:
                            fy_net_inc = annual_inc[report_date].get(
                                "Net Income", None
                            )
                        else:
                            # Fallback: Find closest matching column date
                            closest_col = min(
                                annual_inc.columns,
                                key=lambda x: abs(x - report_date),
                            )
                            # Only use it if it's within 7 days of the balance sheet date
                            if abs((closest_col - report_date).days) <= 7:
                                fy_net_inc = annual_inc[closest_col].get(
                                    "Net Income", None
                                )

                    # Is this row the absolute newest reporting year in the dataframe?
                    # If it is, we map the live TTM and Converted Market Cap directly to it.
                    is_latest = report_date == annual_bs.columns[0]

                    extracted_data.append(
                        {
                            "Ticker": ticker_str,
                            "Year": year_val,
                            "Report Date": date_str,
                            "Trading Currency": trading_curr,
                            "Financial Currency": financial_curr,
                            "Market Cap": (
                                market_cap_converted if is_latest else None
                            ),
                            "Total Assets": total_assets,
                            "Total Current Assets": current_assets,
                            "Total Goodwill and Intangibles": total_goodwill_intangibles,
                            "Total Liabilities": total_liabilities,
                            "Total Investments": total_investments,
                            "Latest FY Net Income": fy_net_inc,
                            "TTM Net Income": ttm_net_inc if is_latest else None,
                        }
                    )
            else:
                print(f"   No annual balance sheet found for {ticker_str}")

        except Exception as e:
            print(f"Error fetching data for {ticker_str}: {e}")

    # Create the DataFrame
    df = pd.DataFrame(extracted_data)

    # Clean up the visual tracking by sorting cleanly by Ticker and Year descending
    if not df.empty:
        df = df.sort_values(
            by=["Ticker", "Year"], ascending=[True, False]
        ).reset_index(drop=True)

    return df

df_old = pull_yf_ticker_data(ticker_list=ticker_list)

df_old

Fetching historical data for: 2267.T...
Fetching historical data for: 3435.T...
Fetching historical data for: 1846.HK...
Fetching historical data for: 6889.HK...


c:\Users\alexa\AppData\Local\Programs\Python\Python310\lib\site-packages\yfinance\scrapers\history.py:389: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  self._capital_gains = pd.Series()


Fetching historical data for: 7399.T...
Fetching historical data for: 1913.HK...


c:\Users\alexa\AppData\Local\Programs\Python\Python310\lib\site-packages\yfinance\scrapers\history.py:389: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  self._capital_gains = pd.Series()


Fetching historical data for: VTU.L...


,Ticker,Year,Report Date,Trading Currency,Financial Currency,Market Cap,Total Assets,Total Current Assets,Total Goodwill and Intangibles,Total Liabilities,Total Investments,Latest FY Net Income,TTM Net Income
0,1846.HK,2025,2025-12-31,HKD,HKD,8.690458e+08,1.952235e+09,7.766520e+08,3.999590e+08,6.677780e+08,NaN,5.445000e+07,5.445000e+07
1,1846.HK,2024,2024-12-31,HKD,HKD,NaN,1.599435e+09,7.136000e+08,2.837040e+08,4.737210e+08,NaN,8.228500e+07,NaN
2,1846.HK,2023,2023-12-31,HKD,HKD,NaN,1.753598e+09,7.878820e+08,3.086520e+08,5.880900e+08,NaN,1.312420e+08,NaN
3,1846.HK,2022,2022-12-31,HKD,HKD,NaN,1.541040e+09,8.383080e+08,2.197010e+08,4.979470e+08,NaN,8.947200e+07,NaN
4,1846.HK,2021,2021-12-31,HKD,HKD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,1913.HK,2025,2025-12-31,HKD,EUR,9.930284e+09,1.101294e+10,3.042250e+09,1.899987e+09,6.347170e+09,NaN,8.519360e+08,8.519360e+08
6,1913.HK,2024,2024-12-31,HKD,EUR,NaN,8.483906e+09,2.493355e+09,8.679200e+08,4.130529e+09,NaN,8.389070e+08,NaN
7,1913.HK,2023,2023-12-31,HKD,EUR,NaN,7.615051e+09,2.162748e+09,8.460240e+08,3.738242e+09,NaN,6.710260e+08,NaN
8,1913.HK,2022,2022-12-31,HKD,EUR,NaN,7.377578e+09,2.424767e+09,8.178090e+08,3.876556e+09,NaN,4.651930e+08,NaN
9,1913.HK,2021,2021-12-31,HKD,EUR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [104]:
ticker_list = ['2267.T', '3435.T', '1846.HK', '6889.HK', '7399.T', '1913.HK', 'VTU.L']

import pandas as pd
import yfinance as yf


def pull_yf_ticker_data(ticker_list: list):
    """Loops through a list of tickers and pulls all available annual data

    from yahoo finance. Returns a multi-row pandas DataFrame structured
    by Ticker and Year.
    """
    extracted_data = []

    for ticker_str in ticker_list:
        print(f"Fetching historical data for: {ticker_str}...")
        try:
            ticker = yf.Ticker(ticker_str)
            info = ticker.info

            # 1. Fetch metadata & currencies
            trading_curr = info.get("currency", "")
            if trading_curr:
                trading_curr = trading_curr.upper()

            financial_curr = info.get("financialCurrency", "")
            if financial_curr:
                financial_curr = financial_curr.upper()

            market_cap = info.get("marketCap", None)

            # Calculate live FX rate if currency mismatch exists
            fx_rate = 1.0
            if (
                trading_curr
                and financial_curr
                and trading_curr != financial_curr
            ):
                fx_ticker_str = f"{trading_curr}{financial_curr}=X"
                try:
                    fx_data = yf.Ticker(fx_ticker_str).history(period="1d")
                    if not fx_data.empty:
                        fx_rate = fx_data["Close"].iloc[-1]
                except Exception:
                    pass

            # Convert live Market Cap if currencies don't match
            market_cap_converted = (
                market_cap * fx_rate if market_cap else None
            )

            # 2. Extract Dedicated Historical Shares Outstanding Series
            shares_series = None
            try:
                shares_series = ticker.get_shares_full(
                    start="2015-01-01", end=None
                )
            except Exception:
                pass

            # 3. Pull Live TTM Net Income (for the current period)
            try:
                ttm_inc = ticker.ttm_income_stmt
                if isinstance(ttm_inc, tuple):
                    ttm_inc = ttm_inc[0]
                if not isinstance(ttm_inc, pd.DataFrame):
                    ttm_inc = pd.DataFrame(ttm_inc)

                ttm_net_inc = (
                    ttm_inc.loc["Net Income"].iloc[0]
                    if not ttm_inc.empty and "Net Income" in ttm_inc.index
                    else None
                )
            except Exception:
                ttm_net_inc = None

            if ttm_net_inc is None:
                try:
                    q_inc = ticker.quarterly_income_stmt
                    if isinstance(q_inc, tuple):
                        q_inc = q_inc[0]
                    if not isinstance(q_inc, pd.DataFrame):
                        q_inc = pd.DataFrame(q_inc)

                    ttm_net_inc = (
                        q_inc.loc["Net Income"].iloc[:4].sum()
                        if not q_inc.empty and "Net Income" in q_inc.index
                        else None
                    )
                except Exception:
                    ttm_net_inc = None

            # 4. Extract ALL Historical Statements & Apply Object Type Safety Guards
            raw_bs = ticker.balance_sheet
            raw_inc = ticker.income_stmt

            if isinstance(raw_bs, tuple):
                raw_bs = raw_bs[0]
            if isinstance(raw_inc, tuple):
                raw_inc = raw_inc[0]

            annual_bs = (
                pd.DataFrame(raw_bs)
                if not isinstance(raw_bs, pd.DataFrame)
                else raw_bs
            )
            annual_inc = (
                pd.DataFrame(raw_inc)
                if not isinstance(raw_inc, pd.DataFrame)
                else raw_inc
            )

            # Check emptiness explicitly via .empty attribute on actual DataFrames
            if not annual_bs.empty:
                # Loop through every reporting date available (columns)
                for report_date in annual_bs.columns:
                    # Create clean string and year markers
                    date_str = str(report_date.date())
                    year_val = report_date.year

                    # Extract balance sheet columns for this specific date
                    bs_col = annual_bs[report_date]

                    total_assets = bs_col.get("Total Assets", None)
                    current_assets = bs_col.get("Current Assets", None)
                    total_liabilities = bs_col.get(
                        "Total Liabilities Net Minority Interest", None
                    )
                    total_investments = bs_col.get(
                        "Investment Properties", None
                    )
                    total_goodwill_intangibles = bs_col.get(
                        "Goodwill And Other Intangible Assets", None
                    )
                    total_equity = bs_col.get("Common Stock Equity", None)

                    # 5. DYNAMIC HISTORICAL MARKET CAP DATA (With Bulletproof Fallbacks)
                    hist_market_cap = None
                    shares_outstanding = None

                    # Strategy A: Try finding historical share series from get_shares_full
                    if shares_series is not None and not shares_series.empty:
                        try:
                            closest_share_date = min(
                                shares_series.index,
                                key=lambda x: abs(
                                    x.tz_localize(None)
                                    - report_date.tz_localize(None)
                                ),
                            )
                            # Only accept it if it matches within a reasonable window (e.g. 180 days)
                            if (
                                abs(
                                    (
                                        closest_share_date.tz_localize(None)
                                        - report_date.tz_localize(None)
                                    ).days
                                )
                                <= 180
                            ):
                                shares_outstanding = shares_series.loc[
                                    closest_share_date
                                ]
                        except Exception:
                            pass

                    # Strategy B: If Series fails or is empty, fall back directly to the raw Balance Sheet line item
                    if shares_outstanding is None or pd.isna(
                        shares_outstanding
                    ):
                        shares_outstanding = bs_col.get(
                            "Ordinary Shares Number", None
                        )

                    # If we managed to resolve a share count via either method, compute historical price close
                    if shares_outstanding and not pd.isna(shares_outstanding):
                        try:
                            start_search = report_date - pd.Timedelta(days=3)
                            end_search = report_date + pd.Timedelta(days=4)

                            price_hist = ticker.history(
                                start=start_search, end=end_search
                            )
                            if not price_hist.empty:
                                price_hist.index = price_hist.index.tz_localize(
                                    None
                                )
                                target_dt = report_date.tz_localize(None)
                                closest_price_idx = min(
                                    price_hist.index,
                                    key=lambda x: abs(x - target_dt),
                                )

                                close_price = price_hist.loc[
                                    closest_price_idx, "Close"
                                ]
                                close_price_converted = close_price * fx_rate
                                hist_market_cap = (
                                    shares_outstanding * close_price_converted
                                )
                        except Exception:
                            pass

                    # Extract matching income statement column for this exact date
                    fy_net_inc = None
                    if not annual_inc.empty:
                        if report_date in annual_inc.columns:
                            fy_net_inc = annual_inc[report_date].get(
                                "Net Income", None
                            )
                        else:
                            closest_col = min(
                                annual_inc.columns,
                                key=lambda x: abs(x - report_date),
                            )
                            if abs((closest_col - report_date).days) <= 7:
                                fy_net_inc = annual_inc[closest_col].get(
                                    "Net Income", None
                                )

                    # Check if this row is the absolute newest reporting year
                    is_latest = report_date == annual_bs.columns[0]

                    # Final assignment tree logic
                    final_market_cap = (
                        market_cap_converted if is_latest else hist_market_cap
                    )
                    if is_latest and final_market_cap is None:
                        final_market_cap = hist_market_cap

                    # CRITICAL FIX: The append is completely out of any conditional check.
                    # Even if final_market_cap is None, the reporting year row is preserved.
                    extracted_data.append(
                        {
                            "Ticker": ticker_str,
                            "Year": year_val,
                            "Report Date": date_str,
                            "Trading Currency": trading_curr,
                            "Financial Currency": financial_curr,
                            "Market Cap": final_market_cap,
                            "Total Assets": total_assets,
                            "Total Current Assets": current_assets,
                            "Total Goodwill and Intangibles": total_goodwill_intangibles,
                            "Total Liabilities": total_liabilities,
                            "Total Equity": total_equity,
                            "Total Investments": total_investments,
                            "Latest FY Net Income": fy_net_inc,
                            "TTM Net Income": (
                                ttm_net_inc if is_latest else None
                            ),
                        }
                    )
            else:
                print(f"   No annual balance sheet found for {ticker_str}")

        except Exception as e:
            print(f"Error fetching data for {ticker_str}: {e}")

    df = pd.DataFrame(extracted_data)

    if not df.empty:
        df = df.sort_values(
            by=["Ticker", "Year"], ascending=[True, False]
        ).reset_index(drop=True)

    return df

In [105]:
def format_currency(val):
    """Formats large numbers into human-readable M (Millions) and B (Billions) strings."""
    # Handle absolute values for the logic, but preserve the negative sign
    abs_val = abs(val)

    if abs_val >= 1_000_000_000_000:
        return f"{val / 1_000_000_000_000:.1f}t"
    elif abs_val >= 1_000_000_000:
        return f"{val / 1_000_000_000:.1f}b"
    elif abs_val >= 1_000_000:
        return f"{val / 1_000_000:.1f}m"
    elif abs_val >= 1_000:
        return f"{val / 1_000:.1f}k"

    return (
        str(int(val)) if val == int(val) else f"{val:.1f}"
    )  # Return as string if small

def format_ratio(val):
    """Returns 'N/A' if the ratio is negative, infinite, or NaN, otherwise formats to 2 decimals."""
    # Check for negative values, NaN (from division by zero), or infinity
    if pd.isna(val) or val < 0 or val == float("inf") or val == float("-inf"):
        return "N/A"
    return f"{val:.2f}x"

def format_percentage(val):
    if pd.isna(val) or val == float("inf") or val == float("-inf"):
        return "N/A"
    return f"{val:.2%}"

In [106]:
df = pull_yf_ticker_data(ticker_list=ticker_list)



def create_calcs(df_):
    # Ensure the dataframe is sorted chronologically by stock and year (ascending)
    # so pandas can accurately calculate multi-year rolling averages
    df_ = df_.sort_values(by=["Ticker", "Year"], ascending=[True, True])

    # 1. First Pass: Create base financial columns and calculate annual ROE as raw floats
    df_ = df_.fillna(0).assign(
        NCAV=lambda df: (df["Total Current Assets"] - df["Total Liabilities"]),
        NCAV_Inv=lambda df: (
            df["Total Current Assets"]
            - df["Total Liabilities"]
            + df["Total Investments"]
        ),
        Book_Value=lambda df: (df["Total Assets"] - df["Total Liabilities"]),
        Tangible_Book_Value=lambda df: (
            df["Total Assets"]
            - df["Total Liabilities"]
            - df["Total Goodwill and Intangibles"]
        ),
        # Using Book_Value as Total Equity. We wrap in a quick replace to avoid dividing by 0
        ROE=lambda df: df["Latest FY Net Income"]
        / df["Book_Value"].replace(0, np.nan),
        Price_Book_Ratio=lambda df: (df["Market Cap"] / df["Book_Value"]),
        Price_Tangible_Book_Ratio=lambda df: (
            df["Market Cap"] / df["Tangible_Book_Value"]
        ),
        Price_NCAV_Ratio=lambda df: (df["Market Cap"] / df["NCAV"]),
        Price_NCAV_Inv_Ratio=lambda df: (df["Market Cap"] / df["NCAV_Inv"]),
        PE_Ratio=lambda df: (df["Market Cap"] / df["Latest FY Net Income"]),
        PE_Ratio_TTM=lambda df: (df["Market Cap"] / df["TTM Net Income"]),
    )

    # Sort back to descending order (Newest year on top) before formatting text strings
    df_ = df_.sort_values(
        by=["Ticker", "Year"], ascending=[True, False]
    ).reindex()

    # --- STRING FORMATTING ZONE ---
    # From this point forward, columns are transformed into readable text strings.

    columns_to_format = [
        "Market Cap",
        "Total Assets",
        "Total Current Assets",
        "Total Liabilities",
        "Total Investments",
        "Total Goodwill and Intangibles",
        "Total Equity",
        "NCAV",
        "NCAV_Inv",
        "Book_Value",
        "Tangible_Book_Value",
        "Latest FY Net Income",
        "TTM Net Income",
    ]
    for col in columns_to_format:
        if col in df_.columns:
            df_[col] = df_[col].map(format_currency)

    columns_to_format_ratios = [
        "Price_Book_Ratio",
        "Price_Tangible_Book_Ratio",
        "Price_NCAV_Ratio",
        "Price_NCAV_Inv_Ratio",
        "PE_Ratio",
        "PE_Ratio_TTM",
    ]
    for col in columns_to_format_ratios:
        if col in df_.columns:
            df_[col] = df_[col].map(format_ratio)

    columns_to_format_perc = ["ROE", "ROE_5YR"]
    for col in columns_to_format_perc:
        if col in df_.columns:
            df_[col] = df_[col].map(format_percentage)

    return df_
    
df_calculated = create_calcs(df)

df_calculated

Fetching historical data for: 2267.T...


c:\Users\alexa\AppData\Local\Programs\Python\Python310\lib\site-packages\yfinance\scrapers\history.py:389: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  self._capital_gains = pd.Series()


Error fetching data for 2267.T: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Fetching historical data for: 3435.T...


c:\Users\alexa\AppData\Local\Programs\Python\Python310\lib\site-packages\yfinance\scrapers\history.py:389: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  self._capital_gains = pd.Series()


Error fetching data for 3435.T: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Fetching historical data for: 1846.HK...


...

Error fetching data for 7399.T: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Fetching historical data for: 1913.HK...


c:\Users\alexa\AppData\Local\Programs\Python\Python310\lib\site-packages\yfinance\scrapers\history.py:389: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  self._capital_gains = pd.Series()
c:\Users\alexa\AppData\Local\Programs\Python\Python310\lib\site-packages\yfinance\scrapers\history.py:389: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  self._capital_gains = pd.Series()
c:\Users\alexa\AppData\Local\Programs\Python\Python310\lib\site-packages\yfinance\scrapers\history.py:389: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  self._capital_gains = pd.Series()


Error fetching data for 1913.HK: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Fetching historical data for: VTU.L...


c:\Users\alexa\AppData\Local\Programs\Python\Python310\lib\site-packages\yfinance\scrapers\history.py:389: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  self._capital_gains = pd.Series()
c:\Users\alexa\AppData\Local\Programs\Python\Python310\lib\site-packages\yfinance\scrapers\history.py:389: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  self._capital_gains = pd.Series()
c:\Users\alexa\AppData\Local\Programs\Python\Python310\lib\site-packages\yfinance\scrapers\history.py:389: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  self._capital_gains = pd.Series()
c:\Users\alexa\AppData\Local\Programs\Python\Python310\lib\site-packages\yfinance\scrape

,Ticker,Year,Report Date,Trading Currency,Financial Currency,Market Cap,Total Assets,Total Current Assets,Total Goodwill and Intangibles,Total Liabilities,...,NCAV_Inv,Book_Value,Tangible_Book_Value,ROE,Price_Book_Ratio,Price_Tangible_Book_Ratio,Price_NCAV_Ratio,Price_NCAV_Inv_Ratio,PE_Ratio,PE_Ratio_TTM
0,1846.HK,2025,2025-12-31,HKD,HKD,869.0m,2.0b,776.7m,400.0m,667.8m,...,108.9m,1.3b,884.5m,4.24%,0.68x,0.98x,7.98x,7.98x,15.96x,15.96x
1,1846.HK,2024,2024-12-31,HKD,HKD,1.3b,1.6b,713.6m,283.7m,473.7m,...,239.9m,1.1b,842.0m,7.31%,1.12x,1.50x,5.26x,5.26x,15.34x,N/A
2,1846.HK,2023,2023-12-31,HKD,HKD,1.7b,1.8b,787.9m,308.7m,588.1m,...,199.8m,1.2b,856.9m,11.26%,1.46x,1.99x,8.54x,8.54x,13.00x,N/A
3,1846.HK,2022,2022-12-31,HKD,HKD,1.7b,1.5b,838.3m,219.7m,497.9m,...,340.4m,1.0b,823.4m,8.58%,1.65x,2.09x,5.06x,5.06x,19.25x,N/A
4,1846.HK,2021,2021-12-31,HKD,HKD,2.5b,0,0,0,0,...,0,0,0,N/A,N/A,N/A,N/A,N/A,N/A,N/A
5,1913.HK,2025,2025-12-31,HKD,EUR,9.9b,11.0b,3.0b,1.9b,6.3b,...,-3.3b,4.7b,2.8b,18.26%,2.13x,3.59x,N/A,N/A,11.66x,11.66x
6,1913.HK,2024,2024-12-31,HKD,EUR,16.5b,8.5b,2.5b,867.9m,4.1b,...,-1.6b,4.4b,3.5b,19.27%,3.79x,4.73x,N/A,N/A,19.67x,N/A
7,2267.T,2025,2025-03-31,JPY,JPY,814.6b,864.3b,377.9b,10.3b,234.8b,...,143.1b,629.5b,619.2b,7.23%,1.29x,1.32x,5.69x,5.69x,17.89x,18.42x
8,3435.T,2025,2025-03-31,JPY,JPY,10.8b,26.6b,15.8b,113.8m,7.8b,...,8.0b,18.8b,18.6b,5.99%,0.57x,0.58x,1.35x,1.35x,9.59x,10.00x
9,6889.HK,2025,2025-03-31,HKD,JPY,44.6b,349.4b,48.0b,7.1b,218.1b,...,-165.3b,131.3b,124.2b,3.05%,0.34x,0.36x,N/A,N/A,11.13x,9.12x
